# FPL Data Explorer
Basic exploration of Fantasy Premier League API data

## Setup & Data Fetching

In [ ]:
import requests
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Fetch bootstrap data (general info about players, teams, etc.)
response = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/')
data = response.json()

print(f"Status: {response.status_code}")
print(f"Available keys: {data.keys()}")

## Players Data

In [ ]:
# Extract players data
players_raw = data['elements']
players = pd.DataFrame(players_raw)

print(f"Total players: {len(players)}")
print(f"\nColumns: {players.columns.tolist()}")
print(f"\nFirst few rows:")
players.head()

In [ ]:
# Get position names
positions = {p['id']: p['singular_name'] for p in data['element_types']}
players['position'] = players['element_type'].map(positions)

# Get team names
teams = {t['id']: t['name'] for t in data['teams']}
players['team'] = players['team'].map(teams)

players.head()

In [ ]:
# Key player stats
print("\nPlayer Statistics:")
print(f"Total Points - Min: {players['total_points'].min()}, Max: {players['total_points'].max()}, Mean: {players['total_points'].mean():.1f}")
print(f"Price Range: £{players['now_cost'].min()/10:.1f}M - £{players['now_cost'].max()/10:.1f}M")
print(f"Average Price: £{players['now_cost'].mean()/10:.1f}M")
print(f"\nPlayers by Position:")
print(players['position'].value_counts())

## Teams Data

In [ ]:
# Extract teams data
teams_df = pd.DataFrame(data['teams'])
print(f"Total teams: {len(teams_df)}")
print(f"\nColumns: {teams_df.columns.tolist()}")
teams_df[['id', 'name', 'strength', 'played', 'wins', 'draws', 'losses']].head(10)

In [ ]:
# Count players per team
players_per_team = players['team'].value_counts().sort_values(ascending=True)
players_per_team.plot(kind='barh')
plt.title('Players per Team')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## Top Players by Points

In [ ]:
# Top 15 players by total points
top_players = players.nlargest(15, 'total_points')[['first_name', 'second_name', 'position', 'team', 'total_points', 'now_cost']]
top_players['price'] = top_players['now_cost'] / 10
top_players = top_players[['first_name', 'second_name', 'position', 'team', 'total_points', 'price']]
top_players

## Price vs Performance

In [ ]:
# Plot: Price vs Total Points
players['price_m'] = players['now_cost'] / 10

fig, ax = plt.subplots(figsize=(12, 6))
for pos in players['position'].unique():
    pos_data = players[players['position'] == pos]
    ax.scatter(pos_data['price_m'], pos_data['total_points'], label=pos, alpha=0.6, s=50)

ax.set_xlabel('Price (£M)')
ax.set_ylabel('Total Points')
ax.set_title('Player Price vs Total Points')
ax.legend()
plt.tight_layout()
plt.show()

## Value Analysis (Points per £M)

In [ ]:
# Calculate points per million pounds
players['points_per_m'] = players['total_points'] / (players['now_cost'] / 10)
players['points_per_m'] = players['points_per_m'].replace([float('inf'), -float('inf')], 0)

# Top value players
best_value = players.nlargest(10, 'points_per_m')[['first_name', 'second_name', 'position', 'team', 'total_points', 'price_m', 'points_per_m']]
best_value.columns = ['First', 'Last', 'Position', 'Team', 'Points', 'Price (£M)', 'Points/£M']
best_value

## Points Distribution by Position

In [ ]:
# Box plot of points by position
fig, ax = plt.subplots(figsize=(10, 6))
players.boxplot(column='total_points', by='position', ax=ax)
ax.set_title('Points Distribution by Position')
ax.set_xlabel('Position')
ax.set_ylabel('Total Points')
plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("\n=== SUMMARY STATISTICS ===")
print(f"\nTotal Players: {len(players)}")
print(f"Total Teams: {teams_df['name'].nunique()}")

print(f"\nPoints by Position:")
position_stats = players.groupby('position')['total_points'].agg(['mean', 'min', 'max', 'count'])
print(position_stats)

print(f"\nPrice (£M) by Position:")
price_stats = players.groupby('position')['price_m'].agg(['mean', 'min', 'max'])
print(price_stats)

## Export Sample Data

In [ ]:
# Save a clean version of the players data
export_cols = ['first_name', 'second_name', 'position', 'team', 'total_points', 'price_m', 'selected_by_percent']
players_export = players[export_cols].copy()
players_export.to_csv('fpl_players.csv', index=False)
print("Exported to fpl_players.csv")

## Player Data Types

In [ ]:
from collections import defaultdict


def get_player_dtypes_from_dataframe(players_df):
    """Extract pandas dtypes from the players DataFrame."""
    return players_df.dtypes.to_dict()


def get_player_dtypes_from_raw(players_raw):
    """Extract native Python/JSON types from raw player objects."""
    type_map = defaultdict(set)

    for player in players_raw:
        for key, value in player.items():
            if value is None:
                type_map[key].add('null')
            else:
                type_map[key].add(type(value).__name__)

    result = {}
    for key, types in type_map.items():
        if len(types) == 1:
            result[key] = list(types)[0]
        else:
            result[key] = list(types)

    return result


def print_player_dtypes_summary(players_raw, players_df):
    """Pretty-print a summary of player data types."""
    raw_types = get_player_dtypes_from_raw(players_raw)
    df_types = get_player_dtypes_from_dataframe(players_df)

    print("=" * 80)
    print("PLAYER DATA TYPES SUMMARY")
    print("=" * 80)
    print(f"\nTotal fields: {len(raw_types)}")
    print(f"Total players: {len(players_raw)}\n")

    print(f"{'Field Name':<40} {'Raw Type(s)':<20} {'Pandas dtype':<15}")
    print("-" * 75)

    for field in sorted(raw_types.keys()):
        raw_type = raw_types[field]
        pandas_type = str(df_types.get(field, 'N/A'))

        if isinstance(raw_type, list):
            raw_type_str = ', '.join(sorted(raw_type))
        else:
            raw_type_str = raw_type

        print(f"{field:<40} {raw_type_str:<20} {pandas_type:<15}")

    print("\n" + "=" * 80)


def inspect_player_sample(players_raw, sample_size=1):
    """Print detailed inspection of sample player(s) with their types."""
    print("\n" + "=" * 80)
    print("SAMPLE PLAYER OBJECT(S)")
    print("=" * 80)

    for i in range(min(sample_size, len(players_raw))):
        player = players_raw[i]
        print(f"\nPlayer {i + 1}: {player.get('web_name', 'N/A')}")
        print("-" * 40)

        for key, value in sorted(player.items()):
            type_name = type(value).__name__ if value is not None else 'NoneType'
            print(f"  {key:<40} {type_name:<12} = {repr(value)[:50]}")


# Print summary
print_player_dtypes_summary(players_raw, players)

# Inspect sample players
inspect_player_sample(players_raw, sample_size=3)